# PoC to summarizer script

La meta para este script es el de poder resumir varios textos largos y condensar la información sin perder información útil.

## Enfoque naive

Se explora un enfoque "naive" donde se chunkeniza el transcript de un lectura de dos horas con distintos tamaños para poder obtener la información más importante de cada chunk. Luego el conjunto de resumenes resultante, vuelve a pasar por un análisis para condensar la información para minimizar la cantidad de resumenes.

### Enfoque por temas

Se explora un enfoque por temas, donde se chunkeniza el transcript de la lectura en chunks más grandes, donde la atención se orienta hacia key-point sentences, que puedan condensar grandes volumenes de información en sentencias muy cortas [definir sentencias cortas].

Estos key-point sentences pueden usarse 

In [1]:
%pip install -qU "langchain[openai]"

Note: you may need to restart the kernel to use updated packages.


In [41]:
import re

class TrancriptLoader:
  def __init__(self, fileName, participants = []):
    self.fileName = fileName
    self.participants = participants

  def load(self) -> list[str]:
    conversation = {}
    with open(self.fileName, 'r') as file:
      content = file.read()
      # splits = re.split(r"\n\n", content)
      splits = re.split(r"\n", content)
      for split in splits:
        if len(self.participants) == 0:
          if 'unknown' not in conversation:
            conversation['unknown'] = []
          conversation['unknown'].append(split)

        for pattern in self.participants:
          if re.match(pattern, split):
            if pattern not in conversation:
              conversation[pattern] = []
            split = re.sub(pattern, '', split).strip()
            conversation[pattern].append(split) 
          else:
            if 'unknown' not in conversation:
              conversation['unknown'] = []
            conversation['unknown'].append(split)
            
      if 'unknown' in conversation:
        return conversation['unknown'] 
      return [conversation[pattern] for pattern in self.participants]
    

In [48]:
# participants = [r'\[Mentor - Freeman Goja\] [0-9]{2}:[0-9]{2}:[0-9]{2}', r'\[Prof\. Uhler\] [0-9]{2}:[0-9]{2}:[0-9]{2}', r'\[Moderator - Ankit Agrawal\] [0-9]{2}:[0-9]{2}:[0-9]{2}']
participants = []
transcript = TrancriptLoader('./sources/Transcript.txt', participants).load()
print(transcript)

['0:00', 'a', '0:31', 'good afternoon or evening or good morning wherever you are in the world', '0:36', 'folks welcome to another episode of show Intel with generative AI here at AWS', '0:42', "today we're going to be talking about a subject that I that I know if you're tuned in right now you want to know all about and that is model context protocol", '0:49', "but before we do that folks I'm Trevor spers I'm a Solutions architect here at AWS I'm joining by two other lovely", '0:55', "Solutions Architects I'm coming in from Salem Massachusetts by the way if you're", '1:00', 'tuned in in the chat let us know where in the world are you coming in from um', '1:06', "these other two amazing essays tell the people who you are why we're here hey yeah thanks Trevor I'm Anil nin I'm", '1:13', "another Sol architect a senior Sol AR here at AWS I'm joining from Princeton New Jersey today um the weather is", '1:20', 'changing every day right yesterday it was in the morning it was like 60 and in', 

In [50]:
"\n".join(transcript)

"0:00\na\n0:31\ngood afternoon or evening or good morning wherever you are in the world\n0:36\nfolks welcome to another episode of show Intel with generative AI here at AWS\n0:42\ntoday we're going to be talking about a subject that I that I know if you're tuned in right now you want to know all about and that is model context protocol\n0:49\nbut before we do that folks I'm Trevor spers I'm a Solutions architect here at AWS I'm joining by two other lovely\n0:55\nSolutions Architects I'm coming in from Salem Massachusetts by the way if you're\n1:00\ntuned in in the chat let us know where in the world are you coming in from um\n1:06\nthese other two amazing essays tell the people who you are why we're here hey yeah thanks Trevor I'm Anil nin I'm\n1:13\nanother Sol architect a senior Sol AR here at AWS I'm joining from Princeton New Jersey today um the weather is\n1:20\nchanging every day right yesterday it was in the morning it was like 60 and in\n1:26\nthe evening it had high 80s uh tod

In [43]:

import dotenv
import getpass
import os

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

In [18]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", model_provider="openai")

In [44]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from constants import SUMMARY_PROMPT

class Summarizer:
  def __init__(self, llm, transcript, chunk_size=3000, chunk_overlap=400):
    self.llm = llm
    text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=chunk_size, chunk_overlap=chunk_overlap, add_start_index=True
    )
    self.chunks = text_splitter.split_text(transcript)

  def summarize(self):
    summaries = []
    for chunk in self.chunks:
      message = self.llm.invoke(SUMMARY_PROMPT.format(CONTENT=chunk))
      summaries.append(message)
    self.summaries = summaries

  def save(self):
    for i, summary in enumerate(self.summaries):
      output_file = f'./Summaries/summary_{i+1}.md'
      
      with open(output_file, 'w') as file:
        file.write(summary.content)

In [51]:
model = Summarizer(llm, "\n".join(transcript), 5000, 400)

In [53]:
len(model.chunks)

13

In [54]:
model.summarize()
model.save()